# CW2 Scaffold C: Volatility Forecasting

Does asymmetric GARCH outperform a symmetric benchmark for equity volatility?

**Module:** FIN306  
**Instructions:** Run provided cells to produce baseline results. Complete the `# TODO` sections. Write a 2,500-word reflective report addressing the five components in the CW2 brief.

---

## What you will do

| Step | Task | Status |
|------|------|--------|
| 1 | Load data and compute log-returns | Provided |
| 2 | Fit GARCH(1,1) baseline | Provided |
| 3 | Fit GJR-GARCH (leverage effect) | **TODO 1** |
| 4 | Rolling walk-forward forecasts | Provided |
| 5 | Mincer-Zarnowitz forecast evaluation | **TODO 2** |
| 6 | Compare GARCH vs GJR-GARCH out-of-sample | **TODO 3** |
| 7 | Diagnostic checks on residuals | **TODO 4** |
| 8 | Extension: GARCH-based VaR backtesting | **TODO 5 (optional)** |

---

## Key concepts

**GARCH(1,1)** decomposes return variance into three components:
$$\sigma^2_t = \omega + \alpha \varepsilon^2_{t-1} + \beta \sigma^2_{t-1}$$

- $\alpha$: how much yesterday's shock feeds into today's variance (ARCH effect)
- $\beta$: how much yesterday's variance persists (GARCH effect)
- $\alpha + \beta$: persistence; if close to 1, shocks die out slowly

**GJR-GARCH** adds a leverage term $\gamma$ to capture asymmetry:
$$\sigma^2_t = \omega + (\alpha + \gamma \cdot \mathbf{1}_{\varepsilon_{t-1}<0})\varepsilon^2_{t-1} + \beta \sigma^2_{t-1}$$

When $\gamma > 0$, negative returns amplify future variance more than positive returns of equal magnitude — the **leverage effect** documented in equity markets.

**Mincer-Zarnowitz (MZ) regression** evaluates forecast quality by regressing realised variance on forecast variance:
$$RV_t = a + b \cdot \widehat{\sigma}^2_t + \varepsilon_t$$

An unbiased, efficient forecast has $a = 0$, $b = 1$, and high $R^2$.

## Before you code: state your research claim

Good data science starts with a question that evidence can refute. Before running any cells, read the claim below and ask yourself the four questions that follow.

---

**Starting claim:**

> *The leverage effect captured by GJR-GARCH is statistically significant in-sample for UK equity returns, but does not produce meaningful out-of-sample forecast improvement over GARCH(1,1), because the additional parameter introduces estimation risk that offsets any model improvement — an empirical instance of the bias-variance tradeoff applied to time-series models.*

---

**Think through these questions before you start:**

1. **Why might in-sample AIC favour GJR-GARCH even when it fails out-of-sample?** What does "estimation risk" mean in this context, and how is it related to the bias-variance tradeoff you studied in Week 7?

2. **What would a meaningful out-of-sample improvement look like?** Think in practical terms: if a risk engine rebalances a portfolio daily, how large does a reduction in forecast error need to be before it justifies the additional operational complexity of a more complex model?

3. **What would falsify this claim?** If GJR-GARCH produces substantially lower MSFE and a better-calibrated Mincer-Zarnowitz regression ($b$ closer to 1, higher $R^2$), what would you conclude? How confident should you be in that conclusion if the difference is small?

4. **Does your conclusion depend on the index chosen?** If you used SPY instead of UKX (or vice versa), would you expect the leverage effect to be stronger or weaker? Why might US and UK equity markets differ on this dimension?

---

Write a one- or two-sentence version of *your own* refined claim in the cell below — before running any code. Your report introduction should open with this claim.

**Your refined claim (write here before running any code):**

*Replace this text with your own one- or two-sentence falsifiable claim. You will return to this cell after completing the analysis and assess whether your results support or refute it.*

## Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# arch library for GARCH estimation
try:
    from arch import arch_model
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'arch', '-q'])
    from arch import arch_model

import statsmodels.api as sm

# Bloomberg loader (works locally and on Colab via fin510-colab-notebooks)
for _scripts_path in [
    Path('scripts'),
    Path('../scripts'),
    Path('resources'),
    Path('../resources'),
]:
    if (_scripts_path / 'bloomberg_loader.py').exists():
        sys.path.insert(0, str(_scripts_path.resolve()))
        break

print('Setup complete.')

## Step 1: Load data

We use UK (UKX) and US (SPY) equity index data from the Bloomberg database. If Bloomberg is not available a simulated dataset is generated so the notebook runs on Colab.

In [ ]:
def _simulate_equity_prices(n=2000, seed=42):
    """Simulate realistic GJR-GARCH-like equity prices as a fallback."""
    rng = np.random.default_rng(seed)
    omega, alpha, gamma, beta = 0.01, 0.05, 0.08, 0.90
    sig2 = np.zeros(n)
    eps  = np.zeros(n)
    sig2[0] = omega / (1 - alpha - gamma/2 - beta)
    for t in range(1, n):
        lev = gamma if eps[t-1] < 0 else 0.0
        sig2[t] = omega + (alpha + lev) * eps[t-1]**2 + beta * sig2[t-1]
        eps[t] = rng.normal(0, np.sqrt(sig2[t]))
    dates = pd.date_range('2015-01-01', periods=n, freq='B')
    prices = pd.Series(100 * np.exp(np.cumsum(eps / 100)), index=dates, name='UKX')
    return pd.DataFrame({'ticker': 'UKX', 'date': dates, 'px_last': prices.values})

TICKERS = ['UKX', 'SPY']
FOCUS   = 'UKX'    # change to 'SPY' if you prefer the US index

try:
    from bloomberg_loader import load_bloomberg
    bb = load_bloomberg(tickers=TICKERS)
    if bb.empty:
        raise FileNotFoundError
    print(f'Bloomberg data loaded: {len(bb)} rows, tickers: {sorted(bb["ticker"].unique())}')
    SIMULATED = False
except (ImportError, FileNotFoundError, Exception):
    print('Bloomberg not available — using simulated GJR-GARCH data.')
    bb = _simulate_equity_prices()
    FOCUS = 'UKX'
    SIMULATED = True

# Extract price series for chosen index
prices = (
    bb[bb['ticker'] == FOCUS]
    .sort_values('date')
    .set_index('date')['px_last']
    .dropna()
)

# Log returns (scaled to percent for numerical stability in arch)
returns = np.log(prices / prices.shift(1)).dropna() * 100
returns.name = f'{FOCUS} log-return (%)'

print(f'\nIndex: {FOCUS} | Observations: {len(returns)}')
print(f'Period: {returns.index[0].date()} to {returns.index[-1].date()}')
print(f'Mean return: {returns.mean():.4f}%  |  Std dev: {returns.std():.4f}%')
if SIMULATED:
    print('\n*** SIMULATED DATA — state clearly in your report ***')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

axes[0].plot(returns.index, returns.values, linewidth=0.5, color='steelblue')
axes[0].axhline(0, color='gray', linewidth=0.7)
axes[0].set_ylabel('Log-return (%)')
axes[0].set_title(f'{FOCUS}: Daily log-returns{" (simulated)" if SIMULATED else ""}')

axes[1].plot(returns.index, returns.abs().values, linewidth=0.5, color='coral')
axes[1].set_ylabel('|Return| (%)')
axes[1].set_title('Absolute returns — proxy for volatility clustering')

plt.tight_layout()
plt.show()

## Step 2: Fit GARCH(1,1) baseline

We split the data: the first 80% is the **in-sample** estimation window; the remaining 20% is held out for walk-forward forecasting.

In [ ]:
split_idx = int(len(returns) * 0.80)
r_train = returns.iloc[:split_idx]
r_test  = returns.iloc[split_idx:]

print(f'Training: {r_train.index[0].date()} to {r_train.index[-1].date()}  ({len(r_train)} obs)')
print(f'Test:     {r_test.index[0].date()}  to {r_test.index[-1].date()}   ({len(r_test)} obs)')

# Fit GARCH(1,1) on training window
garch_spec = arch_model(r_train, vol='Garch', p=1, q=1, mean='Constant')
garch_fit  = garch_spec.fit(disp='off')

print('\n=== GARCH(1,1) in-sample estimates ===')
print(garch_fit.summary().tables[1])

a = garch_fit.params.get('alpha[1]', 0)
b = garch_fit.params.get('beta[1]', 0)
print(f'\nPersistence (α + β): {a + b:.4f}')
print('Values above 0.90 imply slow mean-reversion — shocks take many days to decay.')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(r_train.index, r_train.values, linewidth=0.4, color='steelblue', alpha=0.6, label='Returns')
ax.plot(r_train.index, garch_fit.conditional_volatility, color='crimson', linewidth=1.2, label='GARCH(1,1) σ̂')
ax.plot(r_train.index, -garch_fit.conditional_volatility, color='crimson', linewidth=1.2)
ax.set_title('GARCH(1,1) in-sample conditional volatility')
ax.legend()
plt.tight_layout()
plt.show()

## Step 3: Fit GJR-GARCH (leverage effect)

### TODO 1: Estimate GJR-GARCH and compare parameters

The `arch` library fits GJR-GARCH by setting `vol='GARCH'` and `o=1` (the asymmetry order).

**Your tasks:**

1. Fit GJR-GARCH(1,1) on the **training** window.
2. Print the parameter estimates table.
3. Extract the `gamma[1]` coefficient. Is it positive and statistically significant? (Check the p-value — look for `P>|t|` < 0.05.)
4. Compare AIC/BIC between GARCH and GJR-GARCH. Which model fits better in-sample?
5. Plot GJR-GARCH conditional volatility alongside the GARCH(1,1) estimate.

**In your report:** explain what the leverage effect means economically — why should negative equity shocks raise volatility more than positive shocks of the same magnitude?

In [ ]:
# TODO 1 — your code here
# Hint: arch_model(r_train, vol='Garch', p=1, o=1, q=1, mean='Constant')



## Step 4: Rolling walk-forward forecasts

We generate one-step-ahead variance forecasts for both models by re-estimating on an **expanding window** that grows one observation at a time. This mirrors real-world production use: the model is only trained on data available at forecast time — no look-ahead.

The walk-forward loop takes a few minutes on a long series.

In [ ]:
def walk_forward_forecasts(returns_full, split_idx, vol_type='Garch', o=0):
    """
    Expanding-window walk-forward one-step-ahead variance forecasts.

    Parameters
    ----------
    returns_full : pd.Series
    split_idx    : int — index of first test observation
    vol_type     : 'Garch' for symmetric, 'Garch' with o=1 for GJR
    o            : asymmetry order (0 = GARCH, 1 = GJR-GARCH)

    Returns
    -------
    forecasts : pd.Series of one-step-ahead conditional variance forecasts
                aligned with the test-set dates
    """
    forecasts = []
    dates     = []

    for t in range(split_idx, len(returns_full)):
        window = returns_full.iloc[:t]
        spec   = arch_model(window, vol=vol_type, p=1, o=o, q=1,
                            mean='Constant', rescale=False)
        try:
            res    = spec.fit(disp='off', show_warning=False)
            fc     = res.forecast(horizon=1, reindex=False)
            forecasts.append(fc.variance.values[-1, 0])
        except Exception:
            forecasts.append(np.nan)
        dates.append(returns_full.index[t])

        if (t - split_idx) % 50 == 0:
            pct = (t - split_idx) / (len(returns_full) - split_idx) * 100
            print(f'  Walk-forward: {pct:.0f}% complete', end='\r')

    print(f'  Walk-forward: 100% complete ({len(forecasts)} forecasts)')
    return pd.Series(forecasts, index=dates, name=f'{vol_type}_o{o}_fc_var')


print('Running GARCH(1,1) walk-forward forecasts...')
fc_garch = walk_forward_forecasts(returns, split_idx, vol_type='Garch', o=0)
print('Done.\n')

# Realised variance: squared next-day return (simple proxy)
rv = r_test ** 2
rv.name = 'realised_var'

eval_df = pd.DataFrame({
    'rv':        rv,
    'fc_garch':  fc_garch,
}).dropna()

print(f'Evaluation observations: {len(eval_df)}')
print(eval_df.describe().round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(eval_df.index, np.sqrt(eval_df['rv']),       label='|Realised return|', linewidth=0.6, color='steelblue', alpha=0.7)
ax.plot(eval_df.index, np.sqrt(eval_df['fc_garch']), label='GARCH(1,1) forecast σ̂', linewidth=1.2, color='crimson')
ax.set_title('Walk-forward GARCH(1,1) volatility forecasts vs realised volatility')
ax.set_ylabel('Volatility (%)')
ax.legend()
plt.tight_layout()
plt.show()

## Step 5: Mincer-Zarnowitz forecast evaluation

### TODO 2: Run the MZ regression and interpret

The Mincer-Zarnowitz (MZ) regression tests forecast unbiasedness and efficiency:
$$RV_t = a + b \cdot \widehat{\sigma}^2_t + \varepsilon_t$$

**Your tasks:**

1. Run OLS of `rv` on a constant and `fc_garch` using `statsmodels.api.OLS`.
2. Print the regression summary.
3. Report $a$, $b$, $R^2$, and the p-values.
4. Test the joint hypothesis $a = 0, b = 1$ using an F-test (`res.f_test(...)`).
5. Plot actual `rv` against forecast `fc_garch` (scatter with 45° reference line).

**In your report:** explain what it would mean if $b < 1$ (systematic over-forecasting or under-forecasting of variance); discuss whether a low $R^2$ means the model is useless — compare to the unconditional variance as a naive benchmark.

In [ ]:
# TODO 2 — your code here
# Hint:
#   X = sm.add_constant(eval_df['fc_garch'])
#   mz = sm.OLS(eval_df['rv'], X).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
#   print(mz.summary())
#   F-test: mz.f_test([[0, 1], [1, 0]])  — tests a=0, b=1 jointly



## Step 6: Compare GARCH vs GJR-GARCH out-of-sample

### TODO 3: Run walk-forward GJR-GARCH forecasts and compare

**Your tasks:**

1. Call `walk_forward_forecasts(returns, split_idx, vol_type='Garch', o=1)` to generate GJR-GARCH forecasts. Store as `fc_gjr`.
2. Add `fc_gjr` to `eval_df`.
3. Run the MZ regression for GJR-GARCH forecasts.
4. Compare the two models using:
   - Mean Squared Forecast Error (MSFE): `np.mean((eval_df['rv'] - eval_df['fc_model'])**2)`
   - Mean Absolute Error (MAE): `np.mean(np.abs(eval_df['rv'] - eval_df['fc_model']))`
   - MZ $R^2$
5. Present results in a summary table.
6. Plot both forecast series against realised variance on a single chart.

**In your report:** does in-sample superiority (AIC/BIC) translate to out-of-sample forecast improvement? If not, why might that be? (Think about estimation risk — fitting more parameters on the same data.)

In [ ]:
# TODO 3 — your code here
# Hint: copy the walk_forward call from Step 4, change o=0 to o=1
# Then build a comparison table:
#   pd.DataFrame({'GARCH': [...], 'GJR-GARCH': [...]}, index=['MSFE', 'MAE', 'MZ R²'])



## Step 7: Diagnostic checks

### TODO 4: Standardised residuals from the better model

A well-specified GARCH model should have standardised residuals $\hat{z}_t = \varepsilon_t / \hat{\sigma}_t$ that are:

- Uncorrelated (no autocorrelation in $\hat{z}_t$ or $\hat{z}_t^2$)
- Approximately i.i.d.

**Your tasks:**

1. From the in-sample fit of your preferred model (GARCH or GJR-GARCH), extract `.std_resid`.
2. Plot a 2×2 diagnostic panel:
   - Histogram of standardised residuals (with normal overlay)
   - Q-Q plot
   - ACF of standardised residuals (up to 20 lags)
   - ACF of *squared* standardised residuals
3. Run the Ljung-Box test on squared residuals: `sm.stats.acorr_ljungbox(std_resid**2, lags=[10])`.

**In your report:** if there is still significant autocorrelation in squared residuals, what does that imply? What model extension could address it? (Consider higher-order GARCH, EGARCH, or a different error distribution.)

In [ ]:
# TODO 4 — your code here
# Hint: std_resid = best_fit.std_resid
# from statsmodels.graphics.tsaplots import plot_acf
# from scipy import stats
# For Q-Q: stats.probplot(std_resid, dist='norm', plot=ax)



## Step 8 (extension): GARCH-based Value at Risk

### TODO 5: VaR backtesting *(optional — for higher marks)*

GARCH conditional volatility forecasts feed directly into risk management. A 1-day 99% Value at Risk (VaR) estimate is:
$$\text{VaR}_{0.99,t} = -\hat{\sigma}_t \cdot z_{0.01}$$

where $z_{0.01} = \Phi^{-1}(0.01) \approx -2.326$ under normality.

**Your tasks:**

1. Compute 1-day 99% VaR using the GARCH walk-forward forecast: `var_99 = -np.sqrt(fc_garch) * stats.norm.ppf(0.01)`.
2. Count **VaR breaches**: days when the actual return exceeded (in absolute value) the VaR estimate.
3. The expected breach rate under correct specification is 1%. Compute the actual breach rate. Is it above or below 1%?
4. Plot the return series with VaR band.

**In your report:** discuss why a normally-distributed GARCH VaR tends to underestimate tail risk in equities. What distribution would be more appropriate, and why does this matter for regulatory capital requirements?

In [ ]:
# TODO 5 (optional extension) — your code here
# from scipy import stats
# var_99 = -np.sqrt(eval_df['fc_garch']) * stats.norm.ppf(0.01)
# breaches = (r_test.loc[eval_df.index] < var_99).sum()



---

## Reflective report prompts

Your 2,500-word report must address the five components from the CW2 brief. The prompts below are to help you draft each section.

### A. Method rationale and theoretical foundation (~500 words)

- Why is time-varying volatility more appropriate than a constant-variance assumption for daily equity returns?
- What is volatility clustering, and how do the ACF plots (Step 4 cell 7 of Lab 4 Homework) demonstrate it?
- What is the leverage effect, and why does it motivate GJR-GARCH over the symmetric GARCH?
- What assumptions does GARCH impose? (stationarity: $\alpha + \beta < 1$; error distribution)

### B. Data quality and preparation decisions (~500 words)

- If using Bloomberg data: which index did you choose (UKX or SPY)? Why?
- If using simulated data: state this clearly and explain how it limits your conclusions.
- What are the data quality risks in daily equity price data? (corporate actions, trading halts, stale prices)
- Did you log-transform returns? Why is this standard practice?
- How did you handle the train/test split? What would be wrong with randomly shuffling before splitting?

### C. Results interpretation and validation (~750 words)

- Report and interpret the GARCH parameters ($\omega$, $\alpha$, $\beta$, persistence).
- Does GJR-GARCH find a significant leverage effect ($\gamma > 0$)? What does this imply?
- Interpret the MZ regression: are the forecasts unbiased ($a \approx 0$, $b \approx 1$)? Is $R^2$ high?
- Do GJR-GARCH forecasts outperform GARCH on MSFE/MAE out-of-sample? Does in-sample AIC predict this?

### D. Limitations, assumptions, and potential pitfalls (~500 words)

- Normal innovations underestimate tail risk — what would Student-t GARCH change?
- The MZ regression uses $r_t^2$ as a proxy for realised variance. Why is this noisy? (Contrast with high-frequency realised variance.)
- What is the impact of the 80/20 train-test split choice? Is 20% enough for stable forecasting?
- Regime shifts (structural breaks in volatility) violate stationarity. How would you detect and handle them?

### E. Appropriate use in FinTech contexts and ethics (~250 words)

- Where in FinTech would GARCH volatility forecasts be used? (Risk engines, options pricing, robo-adviser rebalancing triggers)
- What FCA guidance applies to model risk management in algorithmic systems?
- Procyclicality risk: if all institutions use the same GARCH model, volatility spikes can trigger simultaneous de-risking. How is this systemic?
- What does responsible disclosure of model uncertainty look like to a retail investor?